In [ ]:
"""
# Tiny Transformer v53 —— 5m 全日/半日窗

相对 v47：仍 bar5m + 浅 TF/CE/D-stress/e20；默认全日 SEQ=48；半日 `V8C_5M_HALF=1` → SEQ=24。
缓存 `.cache_v49`。提交包: 本 notebook + `transformer_train_v53.py` + `transformer_model_v53.json`
半日训练/推理须同一环境变量。
"""

import os

import numpy as np
import pandas as pd
import dai
import torch
import structlog

from transformer_train_v53 import (
    MODEL_PATH,
    DEFAULT_BAR1M,
    DEFAULT_BAR5M,
    LOCAL_TEST_START,
    LOCAL_TEST_END,
    LOCAL_EVAL_START,
    LOCAL_EVAL_END,
    SCORE_WEIGHTS,
    EndToEndTransformerV53,
    build_infer_tensors,
    pool,
    train_and_save,
    load_model,
    _resolve_tables,
)

logger = structlog.get_logger()

INFER_BATCH = 2048


def main(datasources, start_date, end_date):
    table_1m, table_5m = _resolve_tables(datasources)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch = INFER_BATCH

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"未找到 {MODEL_PATH}; 请先 python transformer_train_v53.py")

    ckpt = load_model(MODEL_PATH, map_location="cpu")
    daily_stats = (
        np.asarray(ckpt["daily_mean"], np.float32),
        np.asarray(ckpt["daily_std"], np.float32),
    )
    intra_stats = (
        np.asarray(ckpt.get("intra_mean", ckpt["mean"]), np.float32),
        np.asarray(ckpt.get("intra_std", ckpt["std"]), np.float32),
    )
    model = EndToEndTransformerV53(**ckpt["model_cfg"])
    model.load_state_dict(ckpt["state_dict"])
    try:
        model = model.to(device)
    except RuntimeError as e:
        logger.warning("model.to(device) 失败，回退 CPU", err=str(e)[:160])
        device = torch.device("cpu")
        batch = 256
        model = model.to(device)
    sw = torch.tensor(
        ckpt.get("score_weights", SCORE_WEIGHTS), dtype=torch.float32, device=device,
    )
    model.eval()
    logger.info(
        "已加载 v53",
        path=MODEL_PATH,
        device=str(device),
        tower=ckpt.get("tower"),
        intra_freq=ckpt.get("intra_freq"),
        seq_intra=ckpt.get("seq_intra"),
        n_params=sum(p.numel() for p in model.parameters()),
    )

    Xi, idx_df = build_infer_tensors(
        table_1m, table_5m, start_date, end_date,
        daily_stats=daily_stats, intra_stats=intra_stats,
        instruments=pool(start_date, end_date),
    )
    preds = []
    Xi_t = torch.from_numpy(np.asarray(Xi, dtype=np.float32))
    with torch.no_grad():
        for i in range(0, len(idx_df), batch):
            xi = Xi_t[i:i + batch].to(device)
            probs = torch.softmax(model(xi).float(), dim=1)
            preds.append((probs * sw).sum(dim=1).cpu().numpy())
    idx_df = idx_df.copy()
    idx_df["score"] = np.concatenate(preds).astype(np.float64)
    idx_df["date"] = pd.to_datetime(idx_df["date"]).dt.normalize()

    stk = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    result = pd.merge(stk, idx_df, on=["date", "instrument"], how="left")
    result = result.replace([np.inf, -np.inf], np.nan)
    miss_before = float(result["score"].isna().mean()) if len(result) else 0.0
    med = result.groupby("date")["score"].transform("median")
    result["score"] = result["score"].fillna(med).fillna(0.0)
    result = (
        result.drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
        .reset_index(drop=True)
    )
    logger.info(
        "分数完成",
        rows=len(result),
        days=int(result["date"].nunique()),
        miss_before_fill=round(miss_before, 4),
    )
    return result


if __name__ == "__main__":
    import sys
    from bigmodule import M

    datasources = {"bar1m": DEFAULT_BAR1M, "bar5m": DEFAULT_BAR5M}
    if not os.path.exists(MODEL_PATH):
        logger.info("开始/续训", path=MODEL_PATH)
        train_and_save(datasources, resume=True)

    start_date, end_date = LOCAL_TEST_START, LOCAL_TEST_END
    eval_start, eval_end = LOCAL_EVAL_START, LOCAL_EVAL_END
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    sys.path.insert(0, "/var/app/enabled/bigmodules/bigalpha_eval")
    import v4.factoranalyze.analyzer as _an
    import v4.factoranalyze.data as _data

    _orig = _an.get_daily_ret

    def _fix(s, e, instruments):
        s2 = (pd.to_datetime(s) + pd.Timedelta(days=30)).strftime("%Y-%m-%d")
        e2 = (pd.to_datetime(e) + pd.Timedelta(days=30)).strftime("%Y-%m-%d")
        return _orig(s2, e2, instruments)

    _an.get_daily_ret = _fix
    if hasattr(_data, "get_daily_ret"):
        _data.get_daily_ret = _fix

    M.bigalpha_eval._latest(
        factor_data=score_data.rename(columns={"score": "factor"}),
        start_date=eval_start,
        end_date=eval_end,
        show=True,
    )
